<a href="https://colab.research.google.com/github/PandeyChhaya/6CS012-AI-ML/blob/main/Worksheet_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6CS012 - Practical Aspects of Training CNN for Image Classification
## Worksheet 6 — Task 1 & Task 2

## Imports

In [8]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Flatten,
    Dropout, BatchNormalization, Activation
)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

---
## Section 1 — Data Understanding and Visualisation

### 1.1 Read Dataset Directory and Extract Class Names

In [9]:
import os

base = "/content/drive/MyDrive"
for root, dirs, files in os.walk(base):
    for d in dirs:
        if "fruit" in d.lower() or "amazon" in d.lower():
            print(os.path.join(root, d))
    if root.count(os.sep) - base.count(os.sep) > 3:
        break

In [10]:
train_dir = "FruitsInAmazon/train"
test_dir  = "FruitsInAmazon/test"

from google.colab import drive
drive.mount('/content/drive')

train_dir = "/content/drive/MyDrive/FruitsInAmazon/train"
test_dir  = "/content/drive/MyDrive/FruitsInAmazon/test"

class_names = [name for name in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, name))]

if not class_names:
    print("No class directories found in the train folder!")
else:
    print(f"Found {len(class_names)} classes: {class_names}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FruitsInAmazon/train'

### 1.2 Check for Corrupted Images

In [ ]:
corrupted_images = []

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            try:
                with Image.open(img_path) as img:
                    img.verify()
            except (IOError, UnidentifiedImageError):
                corrupted_images.append(img_path)

if corrupted_images:
    print("Corrupted Images Found:")
    for img in corrupted_images:
        print(img)
else:
    print("No corrupted images found.")

### 1.3 Class Distribution

In [ ]:
class_counts = {}

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        images = [img for img in os.listdir(class_path)
                  if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        class_counts[class_name] = len(images)

print("Class Distribution:")
print("=" * 45)
print(f"{'Class Name':<25}{'Valid Image Count':>15}")
print("=" * 45)
for class_name, count in class_counts.items():
    print(f"{class_name:<25}{count:>15}")
print("=" * 45)

### 1.4 Visualise Random Images per Class

In [ ]:
selected_images = []
selected_labels = []

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        images = [img for img in os.listdir(class_path)
                  if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if images:
            selected_img = os.path.join(class_path, random.choice(images))
            selected_images.append(selected_img)
            selected_labels.append(class_name)

num_classes = len(selected_images)
cols = (num_classes + 1) // 2
rows = 2

fig, axes = plt.subplots(rows, cols, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < num_classes:
        img = mpimg.imread(selected_images[i])
        ax.imshow(img)
        ax.set_title(selected_labels[i])
        ax.axis("off")
    else:
        ax.axis("off")

plt.tight_layout()
plt.show()

---
## Section 2 — Data Generation and Preprocessing

In [ ]:
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = len(class_names)

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)

### 2.1 Visualise a Training Batch

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(images[i]).astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

### 2.2 Data Augmentation

In [ ]:
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
]

def data_augmentation(images):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images

### 2.3 Visualise Augmented Images

In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    for i in range(9):
        augmented_images = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(augmented_images[0]).astype("uint8"))
        plt.axis("off")
plt.tight_layout()
plt.show()

### 2.4 Performance Optimisation

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

---
## Task 1 — Custom CNN with Batch Normalisation and Dropout

### 3.1 Build the Model

In [ ]:
model = Sequential([
    layers.Lambda(data_augmentation, input_shape=(224, 224, 3)),
    layers.Rescaling(1./255),

    Conv2D(32, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(256, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Flatten(),

    Dense(512, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    Dense(256, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    Dense(128, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    Dense(64, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### 3.2 Train the Model

In [ ]:
history = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
)

### 3.3 Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.4 Evaluate and Generate Classification Report

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\nClassification Report — Custom CNN:")
print(classification_report(y_true, y_pred, target_names=class_names))

### 3.5 Save the Custom CNN Model

In [ ]:
model.save("custom_cnn_fruits.h5")
print("Model saved.")

---
## Task 2 — Transfer Learning with VGG16

### 4.1 Prepare Dataset for VGG16 (Categorical Labels)

In [ ]:
train_ds_tl, val_ds_tl = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
)

train_ds_tl = train_ds_tl.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds_tl   = val_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)

### 4.2 Load Pre-trained VGG16 and Freeze Layers

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

print(f"Total layers in VGG16: {len(base_model.layers)}")
print(f"Trainable layers:      {sum(1 for l in base_model.layers if l.trainable)}")

### 4.3 Add Custom Classification Head

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

vgg_model = Model(inputs=base_model.input, outputs=output)
vgg_model.summary()

### 4.4 Compile and Train VGG16 Model

In [ ]:
vgg_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

vgg_history = vgg_model.fit(
    train_ds_tl,
    epochs=20,
    validation_data=val_ds_tl,
)

### 4.5 Plot VGG16 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(vgg_history.history['accuracy'],     label='Train Accuracy')
axes[0].plot(vgg_history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('VGG16 — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(vgg_history.history['loss'],     label='Train Loss')
axes[1].plot(vgg_history.history['val_loss'], label='Val Loss')
axes[1].set_title('VGG16 — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.6 Evaluate VGG16 and Generate Classification Report

In [ ]:
val_loss_vgg, val_acc_vgg = vgg_model.evaluate(val_ds_tl)
print(f"VGG16 Validation Loss:     {val_loss_vgg:.4f}")
print(f"VGG16 Validation Accuracy: {val_acc_vgg:.4f}")

y_true_vgg = []
y_pred_vgg = []

for images, labels in val_ds_tl:
    preds = vgg_model.predict(images, verbose=0)
    y_true_vgg.extend(np.argmax(labels.numpy(), axis=1))
    y_pred_vgg.extend(np.argmax(preds, axis=1))

print("\nClassification Report — VGG16 Transfer Learning:")
print(classification_report(y_true_vgg, y_pred_vgg, target_names=class_names))

### 4.7 Save VGG16 Model

In [ ]:
vgg_model.save("vgg16_fruits.h5")
print("VGG16 model saved.")

---
## Section 5 — Model Comparison

In [ ]:
print("=" * 50)
print(f"{'Model':<30}{'Val Accuracy':>15}")
print("=" * 50)
print(f"{'Custom CNN':<30}{val_acc:>15.4f}")
print(f"{'VGG16 Transfer Learning':<30}{val_acc_vgg:>15.4f}")
print("=" * 50)

if val_acc_vgg > val_acc:
    print("\nVGG16 outperformed the custom CNN.")
else:
    print("\nCustom CNN matched or outperformed VGG16.")

---
## Section 6 — Inference on Sample Validation Images

In [ ]:
for images, labels in val_ds.take(1):
    sample_images = images[:6]
    sample_labels = labels[:6]

preds_custom = model.predict(sample_images)
pred_labels_custom = np.argmax(preds_custom, axis=1)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_images[i].numpy().astype("uint8"))
    true_label = class_names[int(sample_labels[i])]
    pred_label = class_names[pred_labels_custom[i]]
    color = "green" if true_label == pred_label else "red"
    ax.set_title(f"True: {true_label}\nPred: {pred_label}", color=color)
    ax.axis("off")

plt.suptitle("Custom CNN — Sample Predictions", fontsize=14)
plt.tight_layout()
plt.show()